In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    train_test_split, 
    cross_validate, 
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

In [76]:
RANDOM_STATE = 777

In [77]:
df = pd.read_csv(
    filepath_or_buffer="student_placement_synthetic.csv"
)

In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   branch                     100000 non-null  object 
 1   college_tier               100000 non-null  object 
 2   cgpa                       100000 non-null  float64
 3   backlogs                   100000 non-null  int64  
 4   coding_skills              100000 non-null  float64
 5   dsa_score                  100000 non-null  float64
 6   aptitude_score             100000 non-null  float64
 7   communication_skills       100000 non-null  float64
 8   ml_knowledge               100000 non-null  float64
 9   system_design              100000 non-null  float64
 10  internships                100000 non-null  int64  
 11  projects_count             100000 non-null  int64  
 12  certifications             100000 non-null  int64  
 13  hackathons                 100

`placement_status` for classification, while `salary_package_lpa` is a regression task.

In [79]:
df["branch"].value_counts()

branch
CSE         25046
IT          16065
ECE         14939
EE          12092
ME          12008
CE          10024
Chemical     9826
Name: count, dtype: int64

In [80]:
df["college_tier"].value_counts()

college_tier
Tier-3    45222
Tier-2    39910
Tier-1    14868
Name: count, dtype: int64

tier 1 > tier 2 > tier 3

In [81]:
df["placement_status"].value_counts()

placement_status
1    68475
0    31525
Name: count, dtype: int64

### Encoding

#### One-hot encoding

In [82]:
onehot_encoder = OneHotEncoder(sparse_output=False)
encoded = onehot_encoder.fit_transform(df[["branch"]])

encoded_df = pd.DataFrame(encoded, columns=onehot_encoder.get_feature_names_out())
df = pd.concat([df, encoded_df], axis=1)

df

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,...,extracurriculars,placement_status,salary_package_lpa,branch_CE,branch_CSE,branch_Chemical,branch_ECE,branch_EE,branch_IT,branch_ME
0,ECE,Tier-3,6.70,0,7.6,4.4,49.5,3.7,6.4,0.3,...,1,1,14.75,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,Chemical,Tier-2,5.70,0,5.4,7.9,72.0,8.3,6.3,1.9,...,0,0,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,EE,Tier-2,7.19,0,5.6,6.8,79.1,7.4,4.4,5.2,...,0,1,19.06,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,CE,Tier-2,6.48,0,5.2,3.1,48.4,5.0,1.1,6.7,...,0,0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CSE,Tier-2,6.71,1,5.9,4.7,61.2,4.3,2.7,2.8,...,1,1,13.42,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,IT,Tier-3,6.08,0,3.1,5.4,67.4,4.3,9.1,3.8,...,3,0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0
99996,IT,Tier-3,7.46,0,4.8,6.8,59.5,7.6,5.6,2.4,...,0,1,16.58,0.0,0.0,0.0,0.0,0.0,1.0,0.0
99997,EE,Tier-2,7.94,0,4.9,9.1,55.1,6.1,3.5,5.2,...,2,1,17.24,0.0,0.0,0.0,0.0,1.0,0.0,0.0
99998,ME,Tier-3,6.63,0,5.0,8.0,65.0,5.7,6.2,0.2,...,1,1,17.01,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [83]:
df = df.drop(columns="branch")

#### Ordinal Encoding

In [84]:
ordinal_encoder = OrdinalEncoder(categories=[["Tier-3", "Tier-2", "Tier-1"]])
df["college_tier_encoded"] = ordinal_encoder.fit_transform(df[["college_tier"]])

In [85]:
df

,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,...,placement_status,salary_package_lpa,branch_CE,branch_CSE,branch_Chemical,branch_ECE,branch_EE,branch_IT,branch_ME,college_tier_encoded
0,Tier-3,6.70,0,7.6,4.4,49.5,3.7,6.4,0.3,1,...,1,14.75,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,Tier-2,5.70,0,5.4,7.9,72.0,8.3,6.3,1.9,0,...,0,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,Tier-2,7.19,0,5.6,6.8,79.1,7.4,4.4,5.2,1,...,1,19.06,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,Tier-2,6.48,0,5.2,3.1,48.4,5.0,1.1,6.7,1,...,0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,Tier-2,6.71,1,5.9,4.7,61.2,4.3,2.7,2.8,1,...,1,13.42,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Tier-3,6.08,0,3.1,5.4,67.4,4.3,9.1,3.8,2,...,0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
99996,Tier-3,7.46,0,4.8,6.8,59.5,7.6,5.6,2.4,1,...,1,16.58,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
99997,Tier-2,7.94,0,4.9,9.1,55.1,6.1,3.5,5.2,0,...,1,17.24,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
99998,Tier-3,6.63,0,5.0,8.0,65.0,5.7,6.2,0.2,1,...,1,17.01,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [86]:
df = df.drop(columns="college_tier")

### Model Building

In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   cgpa                       100000 non-null  float64
 1   backlogs                   100000 non-null  int64  
 2   coding_skills              100000 non-null  float64
 3   dsa_score                  100000 non-null  float64
 4   aptitude_score             100000 non-null  float64
 5   communication_skills       100000 non-null  float64
 6   ml_knowledge               100000 non-null  float64
 7   system_design              100000 non-null  float64
 8   internships                100000 non-null  int64  
 9   projects_count             100000 non-null  int64  
 10  certifications             100000 non-null  int64  
 11  hackathons                 100000 non-null  int64  
 12  open_source_contributions  100000 non-null  int64  
 13  extracurriculars           100

In [88]:
X = df.drop(columns=["placement_status", "salary_package_lpa"])
y = df["placement_status"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
)  

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_scaled = scaler.transform(X)
X_test_scaled = scaler.transform(X_test)

#### Simple Logistic Regression

In [89]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

logistic_regression.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.7
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [90]:
y_pred = logistic_regression.predict(X_test_scaled)  
  
print("Accuracy:", accuracy_score(y_test, y_pred))  
print(classification_report(y_test, y_pred))

Accuracy: 0.69915
              precision    recall  f1-score   support

           0       0.57      0.20      0.30      6340
           1       0.72      0.93      0.81     13660

    accuracy                           0.70     20000
   macro avg       0.64      0.57      0.55     20000
weighted avg       0.67      0.70      0.65     20000



In [92]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

scoring = ["accuracy", "f1_weighted", "precision_weighted", "recall_weighted"]

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = cross_validate(
    logistic_regression,
    X_scaled,    
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

for metric in scoring:
    print(f"mean_{metric}: {results['test_' + metric].mean():.4f}")

mean_accuracy: 0.7005
mean_f1_weighted: 0.6494
mean_precision_weighted: 0.6706
mean_recall_weighted: 0.7005


#### Randomized Logistic Regression

In [ ]:
param_grid = {
    "C": np.linspace(start=0.001, stop=1, num=30),
    "solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    "max_iter": [num for num in range(100, 5000, 250)]
}

SyntaxError: invalid non-printable character U+00A0 (1733092051.py, line 2)